In [3]:
import os
import re
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

import faiss  # ✅ needed to read index dimension

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.vectorstores import FAISS


# ============================================================
# CONFIG
# ============================================================
HISTORY_CSV_PATH = r"C:\Users\surya.adatravu\Documents\ContextRAG\RA_FSM_QA.csv"
TEST_CSV_PATH = r"C:\Users\surya.adatravu\Documents\ContextRAG\RA_FSM_RAG_Test_Questions.csv"
INDEX_DIR = r"C:\Users\surya.adatravu\Documents\ContextRAG\vector_db"
SESSION_ID = "pdf_faiss_session"
RESULTS_OUT_CSV = r"C:\Users\surya.adatravu\Documents\ContextRAG\context_rag_test_results.csv"

# Retrieval
USE_MMR = True
TOP_K = 6
FETCH_K = 40
LAMBDA_MULT = 0.3
USE_DISTANCE_GATE = True
DISTANCE_THRESHOLD = 0.85

# Evaluation
VALIDATION_THRESHOLD = 0.82
NORMALIZE_FOR_SCORING = True

# LLM (embedding model will be auto-selected to match index)
LLM_MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)


# ============================================================
# HISTORY STORE
# ============================================================
store = {}

def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


def preload_history_from_csv(session_id: str, csv_path: str, max_rows=None, clear_existing=True):
    df = pd.read_csv(csv_path)
    if not {"Question", "Answer"}.issubset(df.columns):
        raise ValueError(f"History CSV must contain columns Question, Answer. Found: {list(df.columns)}")

    history = get_history(session_id)
    if clear_existing:
        history.clear()

    loaded = 0
    for _, row in df.iterrows():
        if max_rows is not None and loaded >= max_rows:
            break
        history.add_user_message(str(row["Question"]))
        history.add_ai_message(str(row["Answer"]))
        loaded += 1

    print(f"✅ Loaded {loaded} Q/A pairs into history (messages={len(history.messages)}) for session '{session_id}'")
    return history


def load_test_questions(csv_path: str):
    df = pd.read_csv(csv_path)

    possible_q = ["Question", "question", "Query", "query"]
    possible_a = ["Answer", "answer", "Expected", "expected_answer", "GroundTruth", "ground_truth"]

    q_col = next((c for c in possible_q if c in df.columns), None)
    a_col = next((c for c in possible_a if c in df.columns), None)

    if q_col is None:
        raise ValueError(f"Test CSV must include a question column (one of {possible_q}). Found: {list(df.columns)}")
    if a_col is None:
        raise ValueError(f"Test CSV must include an answer column (one of {possible_a}). Found: {list(df.columns)}")

    df = df.rename(columns={q_col: "Question", a_col: "ExpectedAnswer"})
    return df[["Question", "ExpectedAnswer"]]


# ============================================================
# ✅ FIX: LOAD FAISS + AUTO-SELECT EMBEDDINGS TO MATCH INDEX DIM
# ============================================================
def embedding_model_for_faiss_dim(d: int) -> str:
    # Common OpenAI embedding dims (verify against your environment if custom)
    # text-embedding-3-small -> 1536
    # text-embedding-3-large -> 3072
    if d == 1536:
        return "text-embedding-3-small"
    if d == 3072:
        return "text-embedding-3-large"
    raise ValueError(
        f"Unknown FAISS index dimension d={d}. "
        "Cannot auto-select embedding model. Rebuild index or specify a compatible embeddings model."
    )

def load_faiss_and_embeddings(index_dir: str):
    faiss_path = os.path.join(index_dir, "index.faiss")
    pkl_path = os.path.join(index_dir, "index.pkl")

    if not (os.path.exists(faiss_path) and os.path.exists(pkl_path)):
        raise FileNotFoundError(f"FAISS index not found in {index_dir}. Build it first.")

    # Read index to get dimension before creating embeddings
    idx = faiss.read_index(faiss_path)
    d = idx.d

    model_name = embedding_model_for_faiss_dim(d)
    print(f"✅ Detected FAISS dim={d}. Using embeddings model: {model_name}")

    embeddings = OpenAIEmbeddings(model=model_name)

    vs = FAISS.load_local(index_dir, embeddings, allow_dangerous_deserialization=True)
    return vs, embeddings, model_name


# ============================================================
# PROMPTS
# ============================================================
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user's latest question into a standalone search query.\n"
     "Use chat history only to resolve pronouns and references.\n"
     "Return ONLY the rewritten query."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict knowledge assistant.\n"
     "Use chat history only to interpret the question (NOT as knowledge).\n"
     "You must ONLY answer using the provided CONTEXT.\n"
     "IMPORTANT STYLE RULES:\n"
     "- Use exact wording from CONTEXT whenever possible.\n"
     "- Do NOT add extra explanation or commentary.\n"
     "- If the context contains enumerations/lists/levels/states, reproduce them with same labels/order.\n"
     "- Keep the answer concise.\n"
     "If the answer is not in context, say exactly:\n"
     "\"I don't know from provided knowledge.\""),
    MessagesPlaceholder(variable_name="history"),
    ("human",
     "CONTEXT:\n{context}\n\n"
     "QUESTION:\n{question}\n\n"
     "Answer using ONLY the context (follow style rules).")
])

rewrite_chain = rewrite_prompt | llm
rag_chain = rag_prompt | llm

rewrite_with_history = RunnableWithMessageHistory(
    rewrite_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

rag_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="history",
)


# ============================================================
# SCORING / NORMALIZATION
# ============================================================
def normalize_for_scoring(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("I don't know from provided knowledge.", "").strip()
    s = re.sub(r"^\s*\[file=.*?\]\s*$", "", s, flags=re.MULTILINE)
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def similarity_score(text1: str, text2: str, embeddings: OpenAIEmbeddings) -> float:
    v1 = embeddings.embed_query(text1)
    v2 = embeddings.embed_query(text2)
    return float(cosine_similarity([v1], [v2])[0][0])


def validate_pred_vs_expected(pred: str, expected: str, embeddings: OpenAIEmbeddings, threshold: float = VALIDATION_THRESHOLD):
    if NORMALIZE_FOR_SCORING:
        pred = normalize_for_scoring(pred)
        expected = normalize_for_scoring(expected)

    score = similarity_score(pred, expected, embeddings)
    return score >= threshold, score


# ============================================================
# RETRIEVAL (MMR + OPTIONAL DISTANCE GATE)
# ============================================================
def retrieve_docs(vectorstore: FAISS, query: str):
    if USE_MMR:
        docs = vectorstore.max_marginal_relevance_search(
            query, k=TOP_K, fetch_k=FETCH_K, lambda_mult=LAMBDA_MULT
        )
        scored = vectorstore.similarity_search_with_score(query, k=1)
        top_distance = float(scored[0][1]) if scored else None
        return docs, top_distance
    else:
        docs_with_scores = vectorstore.similarity_search_with_score(query, k=TOP_K)
        docs = [d for d, _ in docs_with_scores]
        top_distance = float(docs_with_scores[0][1]) if docs_with_scores else None
        return docs, top_distance


def ask(question: str, vectorstore: FAISS):
    rewritten = rewrite_with_history.invoke(
        {"input": question},
        config={"configurable": {"session_id": SESSION_ID}}
    ).content.strip()

    retrieved_docs, top_distance = retrieve_docs(vectorstore, rewritten)

    if not retrieved_docs:
        return {"question": question, "rewritten_query": rewritten, "answer": "I don't know from provided knowledge.",
                "retrieval_distance": None, "used_sources": ""}

    if USE_DISTANCE_GATE and top_distance is not None and top_distance > DISTANCE_THRESHOLD:
        return {"question": question, "rewritten_query": rewritten, "answer": "I don't know from provided knowledge.",
                "retrieval_distance": top_distance, "used_sources": ""}

    used_sources = sorted({
        f"{d.metadata.get('source_file', '?')}#page={d.metadata.get('page', '?')}"
        for d in retrieved_docs
    })

    context = "\n\n---\n\n".join(
        [
            f"[file={d.metadata.get('source_file','?')} page={d.metadata.get('page','?')}]\n{d.page_content}"
            for d in retrieved_docs
        ]
    )

    resp = rag_with_history.invoke(
        {"question": question, "context": context},
        config={"configurable": {"session_id": SESSION_ID}}
    )

    return {
        "question": question,
        "rewritten_query": rewritten,
        "answer": resp.content.strip(),
        "retrieval_distance": top_distance,
        "used_sources": " | ".join(used_sources),
    }


def run_tests(vectorstore: FAISS, test_df: pd.DataFrame, embeddings: OpenAIEmbeddings):
    rows = []
    pass_count = 0

    for i, r in test_df.iterrows():
        q = str(r["Question"])
        expected = str(r["ExpectedAnswer"])

        out = ask(q, vectorstore)
        pred = out["answer"]

        ok, sim = validate_pred_vs_expected(pred, expected, embeddings)

        rows.append({
            "idx": i,
            "question": q,
            "expected_answer": expected,
            "predicted_answer": pred,
            "cosine_similarity": sim,
            "pass": ok,
            "rewritten_query": out["rewritten_query"],
            "retrieval_distance": out["retrieval_distance"],
            "used_sources": out["used_sources"],
        })

        pass_count += int(ok)
        print(f"[{i+1}/{len(test_df)}] pass={ok} sim={sim:.3f} dist={out['retrieval_distance']}")

    results_df = pd.DataFrame(rows)
    accuracy = pass_count / max(len(test_df), 1)

    print("\n====================")
    print(f"✅ Tests completed: {len(test_df)}")
    print(f"✅ Pass count     : {pass_count}")
    print(f"✅ Accuracy       : {accuracy:.3%}")
    print("====================\n")

    return results_df


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    preload_history_from_csv(SESSION_ID, HISTORY_CSV_PATH, max_rows=None, clear_existing=True)

    vs, emb, emb_model_name = load_faiss_and_embeddings(INDEX_DIR)

    test_df = load_test_questions(TEST_CSV_PATH)
    print(f"✅ Loaded {len(test_df)} test questions from: {TEST_CSV_PATH}")

    results = run_tests(vs, test_df, emb)

    os.makedirs(os.path.dirname(RESULTS_OUT_CSV), exist_ok=True)
    results.to_csv(RESULTS_OUT_CSV, index=False)
    print(f"✅ Saved results to: {RESULTS_OUT_CSV}")


✅ Loaded 50 Q/A pairs into history (messages=100) for session 'pdf_faiss_session'
✅ Detected FAISS dim=1536. Using embeddings model: text-embedding-3-small
✅ Loaded 25 test questions from: C:\Users\surya.adatravu\Documents\ContextRAG\RA_FSM_RAG_Test_Questions.csv
[1/25] pass=False sim=0.282 dist=1.0582702159881592
[2/25] pass=False sim=0.188 dist=0.9055246114730835
[3/25] pass=False sim=0.176 dist=0.9286885261535645
[4/25] pass=False sim=0.175 dist=1.1211011409759521
[5/25] pass=False sim=0.119 dist=1.2967432737350464
[6/25] pass=False sim=0.250 dist=1.3114492893218994
[7/25] pass=False sim=0.569 dist=0.8136768341064453
[8/25] pass=False sim=0.118 dist=1.0272377729415894
[9/25] pass=False sim=0.186 dist=0.8681557774543762
[10/25] pass=False sim=0.184 dist=1.2409489154815674
[11/25] pass=False sim=0.157 dist=0.9988982081413269
[12/25] pass=False sim=0.070 dist=0.9619455337524414
[13/25] pass=False sim=0.203 dist=0.8536734580993652
[14/25] pass=False sim=0.197 dist=1.102959156036377
[15/